# TinyYOLO Experiment Suite — Notebook 7/7
## Model Profiling & ONNX Export (Tables II, III, VI)

**What:** Parameter counts, GFLOPs, ONNX export, and model size for all tables.

**GPU Time:** <1h (profiling only, no training)

**Can run on any machine with PyTorch installed.**

In [ ]:
import os, sys, socket, shutil
from pathlib import Path
REPO = 'https://github.com/ShMazumder/tinyYOLO.git'
try: socket.create_connection(('github.com', 80), timeout=5)
except: print('❌ No internet!'); sys.exit(1)
if Path('tinyYOLO').exists(): shutil.rmtree('tinyYOLO')
!git clone {REPO} --depth 1
%cd tinyYOLO
!pip install -e . -q 2>&1 | tail -1
!pip install tqdm timm thop onnx -q 2>&1 | tail -1
import torch
print('✅ Setup complete!')

In [ ]:
from tinyYOLO.models import build_model
import json, os

print('='*80)
print('MODEL ARCHITECTURE PROFILES')
print('='*80)
print(f'{"Config":<35} {"Params":>10} {"GFLOPs":>10} {"Size(MB)":>10}')
print('-'*80)

profiles = {}
configs = [
    ('det', 'quantized', 20, 'Det-Q (VOC, nc=20)'),
    ('det', 'standard',  20, 'Det-S (VOC, nc=20)'),
    ('det', 'quantized', 80, 'Det-Q (COCO, nc=80)'),
    ('det', 'standard',  80, 'Det-S (COCO, nc=80)'),
    ('seg', 'quantized', 80, 'Seg-Q (COCO, nc=80)'),
    ('seg', 'standard',  80, 'Seg-S (COCO, nc=80)'),
    ('pose','quantized',  1, 'Pose-Q (COCO, nc=1)'),
    ('pose','standard',   1, 'Pose-S (COCO, nc=1)'),
    ('cls', 'quantized',1000,'Cls-Q (ImageNet)'),
    ('obb', 'quantized', 15, 'OBB-Q (DOTA, nc=15)'),
]

for task, variant, nc, label in configs:
    model, info = build_model(task=task, variant=variant, nc=nc)
    total_params = sum(p.numel() for p in model.parameters())
    params_M = round(total_params / 1e6, 2)
    
    # GFLOPs via thop
    try:
        from thop import profile as thop_profile
        x = torch.randn(1, 3, 416, 416)
        flops, _ = thop_profile(model, inputs=(x,), verbose=False)
        gflops = round(flops / 1e9, 2)
    except:
        gflops = 'N/A'
    
    # Model size (FP32)
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
    
    print(f'  {label:<35} {params_M:>8}M {str(gflops):>10} {size_mb:>8.2f}')
    profiles[label] = {'params_M': params_M, 'gflops': gflops, 'size_mb': round(size_mb, 2)}

print('='*80)

# Component breakdown
print('\nCOMPONENT BREAKDOWN (Det-Q, nc=20):')
model, _ = build_model(task='det', variant='quantized', nc=20)
for name, module in [('Backbone', model.backbone), ('Neck', model.neck), ('Head', model.head)]:
    p = sum(p.numel() for p in module.parameters()) / 1e3
    print(f'  {name}: {p:.1f}K params')

os.makedirs('experiments/results', exist_ok=True)
with open('experiments/results/model_profiles.json', 'w') as f:
    json.dump(profiles, f, indent=2)
print('\nSaved: experiments/results/model_profiles.json')

In [ ]:
import glob

# Export best checkpoints to ONNX
checkpoints = sorted(glob.glob('experiments/results/voc-*/best.pt'))
if not checkpoints:
    print('⚠️  No checkpoints found. Run Notebook 01 first for ONNX export.')
    print('    Profiling above is still valid (uses freshly built models).')
else:
    for cp in checkpoints[:2]:  # Export first Q and S checkpoint
        print(f'\n📦 Exporting {cp}...')
        !python scripts/export.py --weights {cp} --imgsz 416

In [ ]:
# Latency profiling (CPU — for GPU latency, use actual edge hardware)
import time
import torch
import numpy as np
from tinyYOLO.models import build_model

print('='*70)
print('INFERENCE LATENCY (CPU, batch=1, 416x416, 100 iterations)')
print('='*70)

for variant, tag in [('quantized', 'Q'), ('standard', 'S')]:
    model, _ = build_model(task='det', variant=variant, nc=20)
    model.eval()
    x = torch.randn(1, 3, 416, 416)
    
    # Warmup
    for _ in range(10):
        with torch.no_grad(): _ = model(x)
    
    # Measure
    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        with torch.no_grad(): _ = model(x)
        times.append((time.perf_counter() - t0) * 1000)
    
    print(f'  TinyYOLO-{tag} (CPU): {np.mean(times):.1f} +/- {np.std(times):.1f} ms')

print('='*70)
print('\nNote: For edge latency numbers (Jetson Nano, RPi4), deploy ONNX models')
print('on actual hardware using TensorRT / TFLite runtimes.')